# Week 002 — Stato incrementale

Periodo: **2026-08-10 — 2026-08-14**

Quattro esercizi progressivi su predicati esatti, aggregazione, sliding window e query online.

## Regole

- Leggi tutti gli esercizi prima di scegliere l'ordine.
- Avvia un timer separato e registra tempi e blocchi in `notes.md`.
- Se superi il timebox, annota il punto raggiunto e passa oltre.
- Non usare AI, soluzioni online o autocomplete generativo.
- Modifica soltanto le celle delle funzioni, non i test.
- Codex valutera' edge case e prestazioni durante la review.


## Mappa della quest

| # | Esercizio | Difficolta' | Timebox | Skill principali |
|---|---|---:|---:|---|
| 1 | Alert confermati per servizio | Facile | 15 min | Predicati esatti, dict |
| 2 | Firma dominante degli incidenti | Media | 20 min | Aggregazione, frequency map, tie-breaking |
| 3 | Batch piu' lungo entro il budget | Media | 30 min | Sliding window, complessita' |
| 4 | Registro dinamico dei modelli | Media | 30 min | Frequency map, query online, stato incrementale |

Tempo target complessivo: **95 minuti**.


# Esercizio 1 — Alert confermati per servizio

**Difficolta':** facile  
**Timebox:** 15 minuti  
**Skill:** predicati esatti, dizionari, record incompleti

Implementa `count_confirmed_alerts(records)`.

Ogni record puo' contenere `service` e `confirmed`. Restituisci un dizionario con una chiave per ogni servizio valido apparso nell'input e il numero dei suoi alert confermati.

Regole:

- ignora i record senza `service` o con `service` uguale a `None`;
- un alert e' confermato soltanto quando `confirmed is True`;
- valori come `1`, stringhe non vuote e liste non vuote non sono il booleano `True` e non devono essere contati;
- un servizio valido deve comparire nel risultato anche se ha zero alert confermati;
- preserva l'ordine di prima apparizione dei servizi;
- non modificare gli input.

### Esempio

```python
count_confirmed_alerts([
    {"service": "api", "confirmed": True},
    {"service": "api", "confirmed": 1},
    {"service": "worker", "confirmed": False},
    {"service": "worker", "confirmed": True},
])
# Output atteso: {"api": 1, "worker": 1}
```


In [ ]:
def count_confirmed_alerts(records):
    """Conta gli alert confermati per ogni servizio valido."""
    raise NotImplementedError("Completa count_confirmed_alerts")

In [ ]:
# Test visibili: non modificarli.
records_test = [
    {"service": "api", "confirmed": True},
    {"service": "api", "confirmed": 1},
    {"service": "worker", "confirmed": False},
    {"service": "worker", "confirmed": True},
    {"service": "batch", "confirmed": "yes"},
    {"confirmed": True},
    {"service": None, "confirmed": True},
]
records_snapshot = [record.copy() for record in records_test]
assert count_confirmed_alerts(records_test) == {"api": 1, "worker": 1, "batch": 0}
assert count_confirmed_alerts([]) == {}
assert count_confirmed_alerts([{"service": "x"}]) == {"x": 0}
assert records_test == records_snapshot
print("Esercizio 1: test visibili superati")

# Esercizio 2 — Firma dominante degli incidenti

**Difficolta':** media  
**Timebox:** 20 minuti  
**Skill:** aggregazione, frequency map, tie-breaking stabile

Implementa `summarize_incidents(incidents)`.

Ogni incidente puo' contenere `service`, `error_code` e `duration`. Restituisci un dizionario indicizzato per servizio. Per ogni servizio calcola:

- `total_incidents`: numero di incidenti validi;
- `total_duration`: somma delle durate;
- `dominant_error`: codice di errore piu' frequente.

Regole:

- un incidente e' valido soltanto se `service` ed `error_code` esistono e non valgono `None`;
- se `duration` manca o vale `None`, considerala zero;
- in caso di parita' tra codici, scegli quello apparso per primo per quel servizio;
- preserva l'ordine di prima apparizione valida dei servizi;
- non modificare gli input;
- l'input puo' contenere fino a 100.000 incidenti.

### Esempio e output atteso

```python
summarize_incidents([
    {"service": "api", "error_code": "E1", "duration": 5},
    {"service": "api", "error_code": "E2", "duration": 3},
    {"service": "api", "error_code": "E1", "duration": None},
    {"service": "worker", "error_code": "W1", "duration": 7},
])
# {
#   "api": {"total_incidents": 3, "total_duration": 8, "dominant_error": "E1"},
#   "worker": {"total_incidents": 1, "total_duration": 7, "dominant_error": "W1"},
# }
```


In [ ]:
def summarize_incidents(incidents):
    """Aggrega incidenti e codice dominante per servizio."""
    raise NotImplementedError("Completa summarize_incidents")

In [ ]:
# Test visibili: non modificarli.
incidents_test = [
    {"service": "api", "error_code": "E1", "duration": 5},
    {"service": "api", "error_code": "E2", "duration": 3},
    {"service": "api", "error_code": "E1", "duration": None},
    {"service": "worker", "error_code": "W1", "duration": 7},
    {"service": "worker", "error_code": "W2"},
    {"service": "worker", "error_code": "W2", "duration": 2},
    {"service": "ignored", "error_code": None, "duration": 99},
]
incidents_snapshot = [record.copy() for record in incidents_test]
assert summarize_incidents(incidents_test) == {
    "api": {"total_incidents": 3, "total_duration": 8, "dominant_error": "E1"},
    "worker": {"total_incidents": 3, "total_duration": 9, "dominant_error": "W2"},
}
assert summarize_incidents([]) == {}
assert summarize_incidents([{"service": "s", "error_code": "B"}, {"service": "s", "error_code": "A"}]) == {
    "s": {"total_incidents": 2, "total_duration": 0, "dominant_error": "B"}
}
assert incidents_test == incidents_snapshot
print("Esercizio 2: test visibili superati")

# Esercizio 3 — Batch piu' lungo entro il budget

**Difficolta':** media  
**Timebox:** 30 minuti  
**Skill:** sliding window, complessita', segmenti contigui

Implementa `longest_batch_within_budget(costs, budget)`.

`costs` e' una lista di interi non negativi. Restituisci la lunghezza massima di un segmento contiguo la cui somma non supera `budget`.

Regole:

- se `budget < 0`, solleva `ValueError`;
- una lista vuota produce zero;
- costi uguali a zero sono ammessi;
- non modificare l'input;
- la funzione deve gestire fino a 200.000 costi.

### Esempio

```python
longest_batch_within_budget([4, 2, 1, 7, 3, 1], 8)
# Output atteso: 3
```


In [ ]:
def longest_batch_within_budget(costs, budget):
    """Restituisce la massima lunghezza contigua entro il budget."""
    raise NotImplementedError("Completa longest_batch_within_budget")

In [ ]:
# Test visibili: non modificarli.
assert longest_batch_within_budget([4, 2, 1, 7, 3, 1], 8) == 3
assert longest_batch_within_budget([], 10) == 0
assert longest_batch_within_budget([0, 0, 0], 0) == 3
assert longest_batch_within_budget([9, 1, 2], 3) == 2
assert longest_batch_within_budget([5, 6], 4) == 0

try:
    longest_batch_within_budget([1, 2], -1)
except ValueError:
    pass
else:
    raise AssertionError("budget < 0 deve sollevare ValueError")

print("Esercizio 3: test visibili superati")

# Esercizio 4 — Registro dinamico dei modelli

**Difficolta':** media  
**Timebox:** 30 minuti  
**Skill:** frequency map, query online, stato incrementale

Implementa `process_model_registry(initial_models, operations)`.

`initial_models` contiene i nomi dei deployment inizialmente attivi, con possibili duplicati. Ogni operazione ha uno dei formati seguenti:

- `["register", model]`: aggiunge un deployment del modello;
- `["retire", model]`: rimuove un deployment, se ne esiste almeno uno;
- `["count", model]`: aggiunge al risultato il numero corrente di deployment del modello;
- `["distinct"]`: aggiunge al risultato il numero corrente di modelli con almeno un deployment.

Regole:

- `retire` su un modello assente non ha effetto;
- i conteggi non possono diventare negativi;
- soltanto `count` e `distinct` producono elementi nella lista di output;
- preserva l'ordine delle risposte;
- non modificare gli input;
- input iniziale e operazioni possono contenere complessivamente fino a 200.000 elementi.

### Esempio

```python
process_model_registry(
    ["alpha", "beta", "alpha"],
    [
        ["count", "alpha"],
        ["distinct"],
        ["retire", "alpha"],
        ["count", "alpha"],
        ["retire", "beta"],
        ["distinct"],
        ["register", "gamma"],
        ["distinct"],
    ],
)
# Output atteso: [2, 2, 1, 1, 2]
```


In [ ]:
def process_model_registry(initial_models, operations):
    """Elabora aggiornamenti e query sul registro dei modelli."""
    raise NotImplementedError("Completa process_model_registry")

In [ ]:
# Test visibili: non modificarli.
models_test = ["alpha", "beta", "alpha"]
operations_test = [
    ["count", "alpha"],
    ["distinct"],
    ["retire", "alpha"],
    ["count", "alpha"],
    ["retire", "beta"],
    ["distinct"],
    ["register", "gamma"],
    ["distinct"],
]
models_snapshot = models_test.copy()
operations_snapshot = [operation.copy() for operation in operations_test]
assert process_model_registry(models_test, operations_test) == [2, 2, 1, 1, 2]
assert process_model_registry([], [["count", "x"], ["distinct"]]) == [0, 0]
assert process_model_registry(["x"], [["retire", "x"], ["retire", "x"], ["count", "x"], ["distinct"]]) == [0, 0]
assert process_model_registry(["x"], [["register", "x"], ["register", "y"], ["count", "x"], ["distinct"]]) == [2, 2]
assert models_test == models_snapshot
assert operations_test == operations_snapshot
print("Esercizio 4: test visibili superati")

# Chiusura della quest

Prima di fermarti:

- esegui tutte le celle nell'ordine corretto;
- verifica che tutti i test visibili passino;
- completa tempi, primo blocco e decisione in `notes.md`;
- non cancellare i tentativi utili: servono per la review.
